In [1]:
states = [
    (d, c, p)
    for d in range(3)
    for c in range(3)
    for p in range(3)
]

print("Number of states:", len(states))
print(states[:10])

Number of states: 27
[(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 1, 0), (0, 1, 1), (0, 1, 2), (0, 2, 0), (0, 2, 1), (0, 2, 2), (1, 0, 0)]


In [2]:
ACTIONS = [0, 1, 2]

HAPTIC_ACTIONS = {
    0: "Low",
    1: "Medium",
    2: "High"
}

In [3]:
import numpy as np

Q = {
    state: np.zeros(3)
    for state in states
}

In [4]:
print(Q[(0, 0, 0)])


[0. 0. 0.]


In [5]:
LEARNING_RATE = 0.10
DISCOUNT_FACTOR = 0.95

EPSILON = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.995

N_EPISODES = 500
STEPS_PER_EPISODE = 50

In [6]:
def choose_action(state, Q, epsilon, rng):

    if rng.random() < epsilon:
        return rng.choice(ACTIONS)

    q_values = Q[state]

    max_q = np.max(q_values)

    best_actions = np.flatnonzero(
        q_values == max_q
    )

    return rng.choice(best_actions)

In [7]:
def performance_level(p):

    if p < 1/3:
        return 0

    elif p < 2/3:
        return 1

    else:
        return 2

In [8]:
def environment_step(
    row,
    performance,
    action,
    rng
):

    ability = row["Learner_Ability"]
    difficulty = row["Difficulty_D"]
    cognitive_effort = row["Cognitive_Effort_C"]

    # Probability without haptic intervention
    p_before = probability_correct(
        ability=ability,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort
    )

    # Probability after selected haptic action
    p_after = probability_correct_with_haptics(
        ability=ability,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort,
        performance=performance,
        action=action
    )

    # Safety clipping
    p_after = np.clip(
        p_after,
        0.0,
        1.0
    )

    # Simulate learner response
    correct = rng.binomial(
        1,
        p_after
    )

    # Reward from Phase 13
    reward = calculate_reward(
        correct=correct,
        p_before=p_before,
        p_after=p_after,
        action=action
    )

    # Update recent performance
    # Simple exponentially-smoothed learner performance
    new_performance = (
        0.8 * performance
        + 0.2 * correct
    )

    new_performance = np.clip(
        new_performance,
        0.0,
        1.0
    )

    return (
        new_performance,
        reward,
        correct,
        p_before,
        p_after
    )

In [14]:
import pandas as pd
import numpy as np

state_df = pd.read_csv(
    "/Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/state_df.csv"
)

print(state_df.shape)
print(state_df.columns.tolist())
print(state_df.head())

(709, 13)
['Excerpt', 'BT.Easiness', 'Predicted_BT_Easiness', 'Difficulty_D', 'Gaze_Score_Raw', 'Cognitive_Effort_C', 'D_level', 'C_level', 'D_label', 'C_label', 'Text_State', 'D_level_q', 'C_level_q']
                                             Excerpt  BT.Easiness  \
0  And outside before the palace a great garden w...    -1.054013   
1  One day he had gone beyond any point which he ...    -1.238432   
2  So off went Lionel to be made a King. He had n...    -0.188186   
3  Father had been away in the country for three ...    -1.009999   
4  It was very dark at first, perhaps because the...    -1.258699   

   Predicted_BT_Easiness  Difficulty_D  Gaze_Score_Raw  Cognitive_Effort_C  \
0              -1.359378      0.570801       -7.549846            0.554339   
1              -0.568831      0.423853      -28.650097            0.325778   
2               0.189702      0.282855      -37.708939            0.227651   
3              -0.477061      0.406794      -22.028875            0.397

In [15]:
rng = np.random.default_rng(42)

episode_rewards = []
episode_accuracy = []

epsilon = EPSILON

In [17]:
import pandas as pd
import numpy as np

simulation_df = pd.read_csv(
    "/Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/state_df.csv"
)

simulation_df["Learner_Profile"] = np.where(
    simulation_df["Predicted_BT_Easiness"] >= 0.67,
    "strong",
    np.where(
        simulation_df["Predicted_BT_Easiness"] >= 0.33,
        "average",
        "struggling"
    )
)

simulation_df["Performance_P"] = simulation_df["Predicted_BT_Easiness"]

print(simulation_df.shape)
print(simulation_df.columns.tolist())
print(
    simulation_df[
        [
            "Excerpt",
            "Predicted_BT_Easiness",
            "Difficulty_D",
            "Cognitive_Effort_C",
            "D_level",
            "C_level",
            "Learner_Profile",
            "Performance_P"
        ]
    ].head()
)

(709, 15)
['Excerpt', 'BT.Easiness', 'Predicted_BT_Easiness', 'Difficulty_D', 'Gaze_Score_Raw', 'Cognitive_Effort_C', 'D_level', 'C_level', 'D_label', 'C_label', 'Text_State', 'D_level_q', 'C_level_q', 'Learner_Profile', 'Performance_P']
                                             Excerpt  Predicted_BT_Easiness  \
0  And outside before the palace a great garden w...              -1.359378   
1  One day he had gone beyond any point which he ...              -0.568831   
2  So off went Lionel to be made a King. He had n...               0.189702   
3  Father had been away in the country for three ...              -0.477061   
4  It was very dark at first, perhaps because the...              -0.832179   

   Difficulty_D  Cognitive_Effort_C  D_level  C_level Learner_Profile  \
0      0.570801            0.554339        2        2      struggling   
1      0.423853            0.325778        1        0      struggling   
2      0.282855            0.227651        0        0      strugglin

In [19]:
print(simulation_df.columns.tolist())

['Excerpt', 'BT.Easiness', 'Predicted_BT_Easiness', 'Difficulty_D', 'Gaze_Score_Raw', 'Cognitive_Effort_C', 'D_level', 'C_level', 'D_label', 'C_label', 'Text_State', 'D_level_q', 'C_level_q', 'Learner_Profile', 'Performance_P']


In [20]:
import pandas as pd
import numpy as np

simulation_df = pd.read_csv(
    "/Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/state_df.csv"
)

simulation_df["Learner_Profile"] = np.where(
    simulation_df["Predicted_BT_Easiness"] >= 0.67,
    "strong",
    np.where(
        simulation_df["Predicted_BT_Easiness"] >= 0.33,
        "average",
        "struggling"
    )
)

simulation_df["Performance_P"] = simulation_df["Predicted_BT_Easiness"]

if "D_level" not in simulation_df.columns:
    simulation_df["D_level"] = simulation_df["D_level_q"]

if "C_level" not in simulation_df.columns:
    simulation_df["C_level"] = simulation_df["C_level_q"]

print(simulation_df.shape)
print(simulation_df.columns.tolist())

print(
    simulation_df[
        [
            "Predicted_BT_Easiness",
            "Difficulty_D",
            "Cognitive_Effort_C",
            "D_level",
            "C_level",
            "Learner_Profile",
            "Performance_P"
        ]
    ].head()
)

(709, 15)
['Excerpt', 'BT.Easiness', 'Predicted_BT_Easiness', 'Difficulty_D', 'Gaze_Score_Raw', 'Cognitive_Effort_C', 'D_level', 'C_level', 'D_label', 'C_label', 'Text_State', 'D_level_q', 'C_level_q', 'Learner_Profile', 'Performance_P']
   Predicted_BT_Easiness  Difficulty_D  Cognitive_Effort_C  D_level  C_level  \
0              -1.359378      0.570801            0.554339        2        2   
1              -0.568831      0.423853            0.325778        1        0   
2               0.189702      0.282855            0.227651        0        0   
3              -0.477061      0.406794            0.397500        1        1   
4              -0.832179      0.472804            0.401087        1        1   

  Learner_Profile  Performance_P  
0      struggling      -1.359378  
1      struggling      -0.568831  
2      struggling       0.189702  
3      struggling      -0.477061  
4      struggling      -0.832179  


In [23]:
# Fix possible column-name mismatches
if "D_level" not in simulation_df.columns and "D_level_q" in simulation_df.columns:
    simulation_df["D_level"] = simulation_df["D_level_q"]

if "C_level" not in simulation_df.columns and "C_level_q" in simulation_df.columns:
    simulation_df["C_level"] = simulation_df["C_level_q"]

if "Performance_P" not in simulation_df.columns and "Predicted_BT_Easiness" in simulation_df.columns:
    simulation_df["Performance_P"] = simulation_df["Predicted_BT_Easiness"]

if "Learner_Profile" not in simulation_df.columns and "Predicted_BT_Easiness" in simulation_df.columns:
    simulation_df["Learner_Profile"] = np.where(
        simulation_df["Predicted_BT_Easiness"] >= 0.67,
        "strong",
        np.where(
            simulation_df["Predicted_BT_Easiness"] >= 0.33,
            "average",
            "struggling"
        )
    )

print(
    simulation_df[
        [
            "Predicted_BT_Easiness",
            "Difficulty_D",
            "Cognitive_Effort_C",
            "D_level",
            "C_level",
            "Learner_Profile",
            "Performance_P"
        ]
    ].head()
)

   Predicted_BT_Easiness  Difficulty_D  Cognitive_Effort_C  D_level  C_level  \
0              -1.359378      0.570801            0.554339        2        2   
1              -0.568831      0.423853            0.325778        1        0   
2               0.189702      0.282855            0.227651        0        0   
3              -0.477061      0.406794            0.397500        1        1   
4              -0.832179      0.472804            0.401087        1        1   

  Learner_Profile  Performance_P  
0      struggling      -1.359378  
1      struggling      -0.568831  
2      struggling       0.189702  
3      struggling      -0.477061  
4      struggling      -0.832179  


In [25]:
for episode in range(N_EPISODES):

    total_reward = 0
    total_correct = 0

    # Start each episode from a random learner-passage row
    row_idx = rng.integers(
        0,
        len(simulation_df)
    )

    row = simulation_df.iloc[row_idx]

    # Initial performance
    performance = float(
        row["Performance_P"]
    )

    for step in range(STEPS_PER_EPISODE):

        # Current discrete state
        state = (
            int(row["D_level"]),
            int(row["C_level"]),
            performance_level(performance)
        )

        # Choose haptic action
        action = choose_action(
            state,
            Q,
            epsilon,
            rng
        )

        # Environment responds
        (
            new_performance,
            reward,
            correct,
            p_before,
            p_after
        ) = environment_step(
            row=row,
            performance=performance,
            action=action,
            rng=rng
        )

        # Sample a new passage for the same learner profile
        learner_rows = simulation_df[
            simulation_df["Learner_Profile"]
            == row["Learner_Profile"]
        ]

        next_row = learner_rows.iloc[
            rng.integers(
                0,
                len(learner_rows)
            )
        ]

        next_state = (
            int(next_row["D_level"]),
            int(next_row["C_level"]),
            performance_level(new_performance)
        )

        # Q-learning update
        old_q = Q[state][action]

        best_next_q = np.max(
            Q[next_state]
        )

        target = (
            reward
            + DISCOUNT_FACTOR * best_next_q
        )

        Q[state][action] = (
            old_q
            + LEARNING_RATE * (target - old_q)
        )

        total_reward += reward
        total_correct += correct

        # Move to next state
        performance = new_performance
        row = next_row

    episode_rewards.append(
        total_reward
    )

    episode_accuracy.append(
        total_correct / STEPS_PER_EPISODE
    )

    # Decay exploration
    epsilon = max(
        EPSILON_MIN,
        epsilon * EPSILON_DECAY
    )

KeyError: 'Learner_Ability'